### Load the Dataset

In [1]:
### Load All Components ────────────
import faiss
import numpy as np
import json
import os
import anthropic
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

CHUNKS_FILE = '../data/processed/chunks.json'
FAISS_FILE  = '../vector_store/faiss_index/alzheimer.index'

print(" Loading components...")

with open(CHUNKS_FILE) as f:
    chunks = json.load(f)
print(f" Chunks:  {len(chunks):,}")

index = faiss.read_index(FAISS_FILE)
print(f" FAISS:   {index.ntotal:,} vectors")

print(" Loading PubMedBERT...")
model = SentenceTransformer(
    'neuml/pubmedbert-base-embeddings'
)
print(f" Model:   PubMedBERT 768-dim")

client = anthropic.Anthropic(
    api_key=os.getenv("ANTHROPIC_API_KEY")
)
print(f" Claude:  API ready!")
print(f"\n All components loaded!")

C:\Users\tpriy\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 Loading components...
 Chunks:  40,995
 FAISS:   40,995 vectors
 Loading PubMedBERT...


Loading weights: 100%|██████████| 199/199 [00:03<00:00, 51.99it/s, Materializing param=pooler.dense.weight]                               


 Model:   PubMedBERT 768-dim
 Claude:  API ready!

 All components loaded!


###  RAG Functions

In [3]:
### RAG Functions ──────────────────
def retrieve_chunks(query, k=5):
    query_vector = model.encode([query])
    query_vector = query_vector.astype('float32')
    distances, indices = index.search(
        query_vector, k=k
    )
    results = []
    for dist, idx in zip(
        distances[0], indices[0]
    ):
        chunk = chunks[idx]
        results.append({
            "title":    chunk['title'],
            "authors":  chunk['authors'],
            "year":     chunk['year'],
            "journal":  chunk['journal'],
            "chunk":    chunk['chunk'],
            "pmid":     chunk['pmid'],
            "distance": float(dist)
        })
    return results

def generate_answer(query, retrieved):
    context = ""
    for i, r in enumerate(retrieved):
        context += f"""
Source {i+1}:
Title:   {r['title']}
Authors: {r['authors']}
Year:    {r['year']}
Content: {r['chunk']}
---"""

    prompt = f"""You are AlzDetect AI — expert
medical research assistant.

Answer using ONLY these PubMed sources.
Cite as (Source 1), (Source 2) etc.
Be specific with biomarker names and findings.
Never hallucinate.

SOURCES:
{context}

QUESTION: {query}"""

    response = client.messages.create(
        model="claude-opus-4-5",
        max_tokens=1000,
        messages=[{
            "role": "user",
            "content": prompt
        }]
    )
    return response.content[0].text

def ask_alzdetect(query):
    retrieved = retrieve_chunks(query, k=5)
    answer    = generate_answer(query, retrieved)
    return retrieved, answer

print(" RAG functions ready!")

 RAG functions ready!


### Test 1: Biomarkers

In [5]:
### Test 1 — Biomarkers ────────────
query = "What are p-tau217 and GFAP biomarkers for Alzheimer's?"

print(f" TEST 1: {query}")
print(f"{'='*55}")

retrieved, answer = ask_alzdetect(query)

print(f"Sources found: {len(retrieved)}")
for i, r in enumerate(retrieved):
    print(f"   {i+1}. {r['title'][:50]}...")
    print(f"      Year: {r['year']} | Distance: {r['distance']:.2f}")

print(f"\n ANSWER:")
print(f"{'='*55}")
print(answer)

 TEST 1: What are p-tau217 and GFAP biomarkers for Alzheimer's?
Sources found: 5
   1. Plasma biomarker profiles in autosomal dominant Al...
      Year: 2023 | Distance: 72.13
   2. Plasma and neuroimaging biomarkers of small vessel...
      Year: 2025 | Distance: 72.79
   3. Timing of changes in Alzheimer's disease plasma bi...
      Year: 2024 | Distance: 75.60
   4. Plasma p-tau231 and p-tau217 as state markers of a...
      Year: 2022 | Distance: 77.81
   5. P-tau217 in Alzheimer's disease....
      Year: 2022 | Distance: 80.75

 ANSWER:
# P-tau217 and GFAP as Biomarkers for Alzheimer's Disease

Based on the provided sources, here are the specific findings for each biomarker:

---

## P-tau217

**Role:** Sensitive and specific biomarker for Alzheimer's disease pathology

**Key Findings:**

- **High specificity for AD:** Plasma p-tau217 is sensitive to clinical manifestation and progression of Alzheimer's disease and is **specific to Alzheimer's disease** — it does not predict patho

### Test 2: Diabetes

In [7]:
# Test 2 — Diabetes ──────────────
query = "How does insulin resistance cause Alzheimer's neurodegeneration?"

print(f" TEST 2: {query}")
print(f"{'='*55}")

retrieved, answer = ask_alzdetect(query)

print(f" Sources found: {len(retrieved)}")
for i, r in enumerate(retrieved):
    print(f"   {i+1}. {r['title'][:50]}...")
    print(f"      Year: {r['year']} | Distance: {r['distance']:.2f}")

print(f"\n ANSWER:")
print(f"{'='*55}")
print(answer)

 TEST 2: How does insulin resistance cause Alzheimer's neurodegeneration?
 Sources found: 5
   1. Brain metabolic dysfunction at the core of Alzheim...
      Year: 2014 | Distance: 113.31
   2. [Mechanisms of neurodegeneration in Alzheimer's di...
      Year: 2012 | Distance: 114.55
   3. Insulin resistance and Alzheimer's disease....
      Year: 2009 | Distance: 117.28
   4. Insulin resistance and pathological brain ageing....
      Year: 2011 | Distance: 117.82
   5. Age-related hyperinsulinemia leads to insulin resi...
      Year: 2019 | Distance: 118.90

 ANSWER:
# Insulin Resistance and Alzheimer's Neurodegeneration

Based on the provided sources, here is how insulin resistance causes Alzheimer's disease neurodegeneration:

## Dual Mechanisms of Brain Insulin Resistance

There are **two pathways** through which insulin resistance leads to AD-type neurodegeneration (Source 3):
1. **Endogenous CNS factors** — originating within the brain itself
2. **Peripheral insulin resistance** —

### Test 3: Treatment

In [9]:
### Test 3 — Treatment ─────────────
query = "How effective is Lecanemab for Alzheimer's treatment?"

print(f" TEST 3: {query}")
print(f"{'='*55}")

retrieved, answer = ask_alzdetect(query)

print(f" Sources found: {len(retrieved)}")
for i, r in enumerate(retrieved):
    print(f"   {i+1}. {r['title'][:50]}...")
    print(f"      Year: {r['year']} | Distance: {r['distance']:.2f}")

print(f"\n ANSWER:")
print(f"{'='*55}")
print(answer)

 TEST 3: How effective is Lecanemab for Alzheimer's treatment?
 Sources found: 5
   1. The Black Book of Psychotropic Dosing and Monitori...
      Year: 2024 | Distance: 98.13
   2. Use of lecanemab for the treatment of Alzheimer's ...
      Year: 2024 | Distance: 104.13
   3. Lecanemab: Looking Before We Leap....
      Year: 2023 | Distance: 105.33
   4. [Disease-modifying Drugs in Alzheimer's Disease: I...
      Year: 2024 | Distance: 106.33
   5. [Disease-modifying therapy with lecanemab for earl...
      Year: 2025 | Distance: 107.01

 ANSWER:
# Effectiveness of Lecanemab for Alzheimer's Treatment

Based on the provided sources, here is a summary of Lecanemab's effectiveness:

## Clinical Efficacy

**Cognitive Decline Reduction:**
- Patients treated with Lecanemab showed **27% less decline** on measures of cognition and function compared to placebo over **18 months** (Source 1)
- The drug demonstrated statistically significant reductions in both **amyloid plaque burden and cognitiv

###  RAG Quality Report

In [10]:
### RAG Quality Report ─────────────
print(f"\n{'='*55}")
print(f" RAG QUALITY REPORT")
print(f"{'='*55}")

test_cases = [
    {
        "query": "blood biomarkers Alzheimer's detection",
        "expected_keywords": ["p-tau", "GFAP", "plasma", "biomarker"]
    },
    {
        "query": "insulin resistance brain Alzheimer's",
        "expected_keywords": ["insulin", "diabetes", "brain", "glucose"]
    },
    {
        "query": "Lecanemab amyloid treatment",
        "expected_keywords": ["Lecanemab", "amyloid", "treatment", "clinical"]
    },
    {
        "query": "APOE4 genetic risk dementia",
        "expected_keywords": ["APOE", "genetic", "risk", "dementia"]
    },
    {
        "query": "deep learning MRI Alzheimer's diagnosis",
        "expected_keywords": ["MRI", "deep learning", "diagnosis", "imaging"]
    }
]

passed = 0
failed = 0

for i, test in enumerate(test_cases):
    retrieved, answer = ask_alzdetect(
        test["query"]
    )
    
    # Check keywords in answer
    keywords_found = sum(
        1 for kw in test["expected_keywords"]
        if kw.lower() in answer.lower()
    )
    
    best_distance = retrieved[0]['distance']
    status = " PASS" if keywords_found >= 2 else " FAIL"
    
    if keywords_found >= 2:
        passed += 1
    else:
        failed += 1

    print(f"\nTest {i+1}: {test['query'][:40]}...")
    print(f"   Status:      {status}")
    print(f"   Keywords:    {keywords_found}/{len(test['expected_keywords'])} found")
    print(f"   Distance:    {best_distance:.2f}")
    print(f"   Best source: {retrieved[0]['title'][:45]}...")

print(f"\n{'='*55}")
print(f" FINAL SCORE:")
print(f"   Passed: {passed}/5")
print(f"   Failed: {failed}/5")
print(f"   Score:  {passed/5*100:.0f}%")
print(f"\n{'='*55}")
if passed >= 4:
    print(f" RAG PIPELINE: PRODUCTION READY!")
else:
    print(f" RAG PIPELINE: NEEDS IMPROVEMENT")
print(f"{'='*55}")



 RAG QUALITY REPORT

Test 1: blood biomarkers Alzheimer's detection...
   Status:       PASS
   Keywords:    3/4 found
   Distance:    89.66
   Best source: Plasma biomarkers for Alzheimer's and related...

Test 2: insulin resistance brain Alzheimer's...
   Status:       PASS
   Keywords:    3/4 found
   Distance:    77.76
   Best source: Insulin Signaling in Alzheimer's Disease: Ass...

Test 3: Lecanemab amyloid treatment...
   Status:       PASS
   Keywords:    4/4 found
   Distance:    85.58
   Best source: Lecanemab: A Second in Class Therapy for the ...

Test 4: APOE4 genetic risk dementia...
   Status:       PASS
   Keywords:    4/4 found
   Distance:    88.88
   Best source: Leading determinants of incident dementia amo...

Test 5: deep learning MRI Alzheimer's diagnosis...
   Status:       PASS
   Keywords:    4/4 found
   Distance:    67.24
   Best source: Advanced MRI based Alzheimer's diagnosis thro...

 FINAL SCORE:
   Passed: 5/5
   Failed: 0/5
   Score:  100%

 RAG PIPEL